# JobMatch AI — Pipeline de Composição dos Datasets

Este notebook executa e valida cada etapa da pipeline de composição:

1. Normalização de salários (pay_period → anual USD)
2. Criação do campo `full_text` (title + description + skills_desc)
3. Fuzzy match de títulos para mapear skills (rapidfuzz)
4. Composição final e salvamento em Parquet
5. Quality check

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Garantir que o diretório raiz do projeto está no path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import json
import os

print(f"📂 Projeto raiz: {project_root}")
print(f"🐍 Python: {sys.version}")

---
## 0. Carregar Dados Brutos

Tenta carregar de cache (Parquet). Se não existir, dispara `load_all()` — requer Kaggle API configurada.

In [ ]:
from src.pipeline.load_data import load_all
from src.pipeline.compose_datasets import (
    normalize_salary,
    add_full_text,
    build_skills_map,
    compose,
    quality_check,
)
from src.pipeline.preprocess import clean_text

RAW_DIR = project_root / "data" / "raw"
PROCESSED_DIR = project_root / "data" / "processed"

In [ ]:
# Verificar se já existem dados processados em cache
cached_jobs = PROCESSED_DIR / "jobs_clean.parquet"
cached_pairs = PROCESSED_DIR / "training_pairs.parquet"

if cached_jobs.exists() and cached_pairs.exists():
    print("📦 Dados processados encontrados em cache. Carregando...")
    jobs_df = pd.read_parquet(cached_jobs)
    pairs_df = pd.read_parquet(cached_pairs)
    print(f"   jobs_df: {jobs_df.shape}")
    print(f"   pairs_df: {pairs_df.shape}")
else:
    print("📦 Nenhum cache encontrado. Executando download completo...")
    print("⚠️  Requer Kaggle API configurada (kaggle.json em ~/.kaggle/)")
    raw = load_all()
    jobs_df = raw['linkedin']
    pairs_df = raw['resume_jd']
    skills_df = raw['job_skills']
    print(f"\n📊 LinkedIn: {jobs_df.shape}")
    print(f"📊 Resume-JD: {pairs_df.shape}")
    print(f"📊 Skills: {skills_df.shape}")

---
## 1. Normalização de Salários

Converte `min_salary` / `max_salary` para valores anuais em USD com base em `pay_period`.

Fatores de conversão:
- HOURLY → × 2.080 (40h × 52 semanas)
- DAILY  → × 260 (5 dias × 52 semanas)
- WEEKLY → × 52
- MONTHLY → × 12
- YEARLY → × 1

In [ ]:
print("📊 Colunas de salário disponíveis:")
salary_cols = [c for c in jobs_df.columns if 'salary' in c.lower()]
print(f"   {salary_cols}")

if salary_cols:
    amostra = jobs_df[salary_cols].dropna(how='all').head(5)
    display(amostra)

In [ ]:
jobs_normalized = normalize_salary(jobs_df.copy())

print("\n📊 Após normalização:")
new_cols = ['min_salary_annual', 'max_salary_annual', 'salary_annual_avg']
display(jobs_normalized[new_cols].describe())

if 'pay_period' in jobs_normalized.columns:
    print("\n📊 Distribuição de pay_period:")
    display(jobs_normalized['pay_period'].value_counts())

---
## 2. Criação do Campo full_text

Combina `title` + `description` + `skills_desc` em uma única string limpa via `clean_text()`.

In [ ]:
# Testar clean_text em um exemplo
exemplo = jobs_df.iloc[0]['description'] if 'description' in jobs_df.columns else ""
if isinstance(exemplo, str) and len(exemplo) > 50:
    print("🔤 Texto original (primeiros 500 chars):")
    print(exemplo[:500])
    print("\n🔤 Após clean_text:")
    print(clean_text(exemplo)[:500])
else:
    print("⚠️  Descrição de exemplo muito curta ou ausente. Usando texto genérico...")
    print(clean_text("Data Scientist with 5 years of experience in Python, SQL, and Machine Learning."))

In [ ]:
jobs_with_text = add_full_text(jobs_df.copy())

print("\n📊 Estatísticas do full_text:")
tamanhos = jobs_with_text['full_text'].str.split().str.len()
print(f"   Média de tokens: {tamanhos.mean():.0f}")
print(f"   Mediana de tokens: {tamanhos.median():.0f}")
print(f"   Min tokens: {tamanhos.min()}")
print(f"   Max tokens: {tamanhos.max()}")

print("\n📊 Amostra de full_text:")
display(jobs_with_text[['title', 'full_text']].head(3))

---
## 3. Fuzzy Match de Skills (rapidfuzz)

Para cada título único de vaga, encontra o título mais próximo no Job Skills Dataset usando `token_sort_ratio` com threshold 80.

In [ ]:
# Verificar se skills_df existe ou carregar do cache
if 'skills_df' not in dir() or skills_df is None:
    skills_cache = list(RAW_DIR.glob("*skill*.csv")) or list(RAW_DIR.glob("*.csv"))
    if skills_cache:
        skills_df = pd.read_csv(skills_cache[0], low_memory=False)
        print(f"📊 Skills carregado de cache: {skills_df.shape}")
        display(skills_df.head(2))
    else:
        print("⚠️  Skills dataset não encontrado. Usando dados simulados de demonstração.")
        skills_df = pd.DataFrame({
            'job_title': ['data scientist', 'data analyst', 'data engineer',
                          'machine learning engineer', 'software engineer'],
            'skills': [
                'python, sql, machine learning, statistics, deep learning',
                'python, sql, excel, tableau, power bi',
                'python, sql, spark, airflow, docker, aws',
                'python, tensorflow, pytorch, mlops, docker, kubernetes',
                'java, python, javascript, docker, kubernetes, aws',
            ]
        })

In [ ]:
print(f"🔗 Títulos únicos no LinkedIn: {jobs_with_text['title'].nunique():,}")
print(f"🔗 Títulos no Skills Dataset: {skills_df['job_title'].nunique()}")

skills_map = build_skills_map(jobs_with_text, skills_df, score_threshold=80)

print("\n📊 Amostra do skills_map:")
sample_items = list(skills_map.items())[:5]
for title, skills in sample_items:
    print(f"   {title}: {skills}")

---
## 4. Composição Final

Executa a pipeline completa e salva os DataFrames processados em `data/processed/`.

In [ ]:
raw_dict = {
    'linkedin': jobs_df,
    'resume_jd': pairs_df,
    'job_skills': skills_df if 'skills_df' in dir() else pd.DataFrame(),
}

jobs_final, pairs_final = compose(raw_dict, output_dir=PROCESSED_DIR)

---
## 5. Quality Check

Verifica qualidade dos dados processados antes do treinamento.

In [ ]:
quality_check(jobs_final, pairs_final)

---
## 6. Validação das Colunas Essenciais

Verifica se todas as colunas necessárias para os modelos estão presentes.

In [ ]:
required_jobs_cols = [
    'title', 'full_text', 'min_salary_annual',
    'max_salary_annual', 'salary_annual_avg', 'required_skills'
]

required_pairs_cols = ['resume', 'job_description', 'label']

print("📋 Validação de colunas — Jobs:")
for col in required_jobs_cols:
    if col in jobs_final.columns:
        n_non_null = jobs_final[col].notna().sum() if col != 'required_skills' else (jobs_final[col].str.len() > 0).sum()
        print(f"   ✅ {col}: {n_non_null:,} não nulos")
    else:
        print(f"   ❌ {col}: AUSENTE")

print("\n📋 Validação de colunas — Pairs:")
for col in required_pairs_cols:
    if col in pairs_final.columns:
        print(f"   ✅ {col}: {pairs_final[col].notna().sum():,} não nulos")
    else:
        print(f"   ❌ {col}: AUSENTE")

---
## 7. Limpeza de Memória (Opcional)

Remove DataFrames intermediários para liberar RAM.

In [ ]:
del jobs_normalized, jobs_with_text
import gc; gc.collect()
print("🧹 Memória liberada")

---
## Resumo

| Etapa | Status |
|-------|--------|
| 1. Normalização de salários | ✅ |
| 2. Criação de full_text | ✅ |
| 3. Fuzzy match de skills | ✅ |
| 4. Composição final | ✅ |
| 5. Quality check | ✅ |
| 6. Validação de colunas | ✅ |

Os dados processados estão em `data/processed/`:
- `jobs_clean.parquet` — Vagas limpas e enriquecidas
- `training_pairs.parquet` — Pares currículo-vaga para classificador
- `skills_map.json` — Mapeamento título → skills